# Sports14 Text/Image Feature Extraction

In [1]:

import os
import numpy as np
import pandas as pd

In [2]:
os.chdir('../data/grocery')
os.getcwd()

'/home/hun/MMRec_AlignRec/data/grocery'

## Load text data

In [3]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-grocery.csv'

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (8713, 10)


,itemID,asin,description,title,imUrl,related,salesRank,categories,price,brand
0,0,616719923X,Green Tea Flavor Kit Kat have quickly become t...,Japanese Kit Kat Maccha Green Tea Flavor (5 Ba...,http://ecx.images-amazon.com/images/I/51LdEao6...,"{'also_bought': ['B00FD63L5W', 'B0047YG5UY', '...",{'Grocery & Gourmet Food': 37305},[['Grocery & Gourmet Food']],NaN,NaN
1,1,9742356831,Used to make various curry soups and stir fry ...,Mae Ploy Thai Green Curry Paste - 14 oz jar,http://ecx.images-amazon.com/images/I/41pQp67A...,"{'also_bought': ['B000EI2LLO', 'B000EICISA', '...",{'Grocery & Gourmet Food': 3434},[['Grocery & Gourmet Food']],7.23,Mae Ploy
2,2,B00004S1C5,"From Easter eggs to colorful cookies, Spectrum...","Ateco Food Coloring Kit, 6 colors",http://ecx.images-amazon.com/images/I/41F75K9F...,"{'also_bought': ['B0000CFMLT', 'B002PO3KBK', '...",{'Kitchen & Dining': 4494},"[['Grocery & Gourmet Food', 'Cooking & Baking'...",9.76,HIC Harold Import Co.


In [4]:

# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(13, 10)
(1204, 10)
(13, 10)
(13, 10)
(0, 10)


In [5]:

df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


In [6]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row[desc_str]
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:10]

["Japanese Kit Kat Maccha Green Tea Flavor (5 Bag) (4.91oz x 5)   Grocery & Gourmet Food Green Tea Flavor Kit Kat have quickly become the most sought after snack from Japan. Here are a limited supply of Maccha Green Tea flavored Kit Kat that you would not want to miss. These epic snacks have a sweet maccha green tea flavor mixed with creamy white chocolate, on a crispy wafer that Nestle's has perfected. Each bag contains 12 individually wrapped bars.",
 'Mae Ploy Thai Green Curry Paste - 14 oz jar Mae Ploy Grocery & Gourmet Food Used to make various curry soups and stir fry dishes. Mae Ploy Brand is recognized in Thailand as a high quality export product with rich taste and authentic flavor. Packed in convenient plastic tub with tighly sealed lid. All natural.',
 "Ateco Food Coloring Kit, 6 colors HIC Harold Import Co. Grocery & Gourmet Food Cooking & Baking Food Coloring From Easter eggs to colorful cookies, Spectrum's gel food colorings provide a wide range of decorative touches, wit

In [7]:

course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [8]:
# should `pip install sentence_transformers` first
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentence_embeddings = model.encode(sentences)
print('text encoded!')

assert sentence_embeddings.shape[0] == df.shape[0]
np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
print('done!')


/home/hun/miniconda3/envs/ipy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hun/miniconda3/envs/ipy/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/hun/miniconda3/envs/ipy/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


text encoded!
done!


In [9]:
sentence_embeddings[:10]

array([[ 0.01657731, -0.00646653,  0.05703938, ..., -0.04410642,
        -0.01208084, -0.02984802],
       [-0.03333354, -0.03070537,  0.06911523, ...,  0.0018233 ,
         0.0642586 , -0.00366994],
       [-0.03570388, -0.01664365,  0.00453851, ...,  0.01162007,
         0.00400354, -0.02355654],
       ...,
       [-0.00148825,  0.02671757,  0.06408443, ..., -0.03550781,
         0.12077958,  0.00370013],
       [-0.05085435, -0.13789088, -0.03795417, ..., -0.0008477 ,
         0.11135017,  0.04063827],
       [ 0.0018889 , -0.05882647,  0.05027296, ..., -0.09206817,
         0.06217723, -0.11353085]], shape=(10, 384), dtype=float32)

In [10]:
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

(8713, 384)


array([[ 0.01657731, -0.00646653,  0.05703938, ..., -0.04410642,
        -0.01208084, -0.02984802],
       [-0.03333354, -0.03070537,  0.06911523, ...,  0.0018233 ,
         0.0642586 , -0.00366994],
       [-0.03570388, -0.01664365,  0.00453851, ...,  0.01162007,
         0.00400354, -0.02355654],
       ...,
       [-0.00148825,  0.02671757,  0.06408443, ..., -0.03550781,
         0.12077958,  0.00370013],
       [-0.05085435, -0.13789088, -0.03795417, ..., -0.0008477 ,
         0.11135017,  0.04063827],
       [ 0.0018889 , -0.05882647,  0.05027296, ..., -0.09206817,
         0.06217723, -0.11353085]], shape=(10, 384), dtype=float32)

# Image encoder (V0)，following LATTICE, averaging over for missed items

In [11]:
df[:5]

,itemID,asin,description,title,imUrl,related,salesRank,categories,price,brand
0,0,616719923X,Green Tea Flavor Kit Kat have quickly become t...,Japanese Kit Kat Maccha Green Tea Flavor (5 Ba...,http://ecx.images-amazon.com/images/I/51LdEao6...,"{'also_bought': ['B00FD63L5W', 'B0047YG5UY', '...",{'Grocery & Gourmet Food': 37305},[['Grocery & Gourmet Food']],NaN,
1,1,9742356831,Used to make various curry soups and stir fry ...,Mae Ploy Thai Green Curry Paste - 14 oz jar,http://ecx.images-amazon.com/images/I/41pQp67A...,"{'also_bought': ['B000EI2LLO', 'B000EICISA', '...",{'Grocery & Gourmet Food': 3434},[['Grocery & Gourmet Food']],7.23,Mae Ploy
2,2,B00004S1C5,"From Easter eggs to colorful cookies, Spectrum...","Ateco Food Coloring Kit, 6 colors",http://ecx.images-amazon.com/images/I/41F75K9F...,"{'also_bought': ['B0000CFMLT', 'B002PO3KBK', '...",{'Kitchen & Dining': 4494},"[['Grocery & Gourmet Food', 'Cooking & Baking'...",9.76,HIC Harold Import Co.
3,3,B0000531B7,,"PowerBar Harvest Energy Bars, Double Chocolate...",http://ecx.images-amazon.com/images/I/519SuVj1...,"{'also_bought': ['B000EC63PU', 'B00DZGEY44', '...",{'Grocery & Gourmet Food': 2858},[['Grocery & Gourmet Food']],24.75,Powerbar
4,4,B00005344V,"For nearly forty years, we&#x2019;ve been pass...","Traditional Medicinals Breathe Easy, 16-Count ...",http://ecx.images-amazon.com/images/I/51H54cd-...,"{'also_bought': ['B0009F3POE', 'B0009F3POO', '...",{'Grocery & Gourmet Food': 5034},[['Grocery & Gourmet Food']],21.74,Traditional Medicinals


In [12]:
import array

def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin = f.read(10).decode('UTF-8')
    if asin == '': break
    a = array.array('f')
    a.fromfile(f, 4096)
    yield asin, a.tolist()

In [13]:

img_data = readImageFeatures("image_features_Grocery_and_Gourmet_Food.b")
item2id = dict(zip(df['asin'], df['itemID']))

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])
avg = np.array(avg).mean(0).tolist()

ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i])
    else:
        non_no.append(i)
        ret.append(avg)

print('# of items not in processed image features:', len(non_no))
assert len(ret) == len(item2id)
np.save('image_feat.npy', np.array(ret))
np.savetxt("missed_img_itemIDs.csv", non_no, delimiter =",", fmt ='%d')
print('done!')

# of items not in processed image features: 97
done!
